# Train E(n) Equivariant GNN on LP-PDBBind

This notebook trains the custom `GNNPredictor` architecture from the Virtual Lab Manager (VLM) repository on the LP-PDBBind dataset.

**Hardware:** Requires a GPU runtime (A100 recommended) to process 19K 3D molecular graphs and ESM-2 embeddings.

## 1. Environment Setup

In [ ]:
# Install dependencies (Colab has torch, we need PyG and RDKit)
!pip install -q torch-geometric rdkit-pypi transformers accelerate

## 2. Clone VLM Repository & Upload Data

> **ACTION REQUIRED:** Once the repo is cloned, you must upload the LP-PDBBind data to `VLM/data/pdbbind_deepchem/raw/LP_PDBBind.csv` in the Colab file browser on the left.

In [ ]:
import os
import sys

# In a real scenario, you would git clone your VLM repo here.
# !git clone https://github.com/YOUR-USERNAME/VLM.git
# %cd VLM

print("Ensure your working directory is the VLM root and LP_PDBBind.csv is in data/pdbbind_deepchem/raw/")

## 3. Dataset Preprocessing

In [ ]:
import torch
from torch_geometric.loader import DataLoader

# 1. This will parse the CSV, convert SMILES to 3D PyG graphs via RDKit,
# and hit the ESM-2 API/local model to generate protein pocket embeddings.
# This step takes time but caches the result in data/pdbbind_deepchem/processed/
from data.lp_pdbbind import LPPDBBind

dataset = LPPDBBind(root='data/pdbbind_deepchem')
print(f"Dataset loaded with {len(dataset)} pairs.")

# 2. Shuffle and split
dataset = dataset.shuffle()
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))

train_dataset = dataset[:train_size]
val_dataset = dataset[train_size:train_size + val_size]
test_dataset = dataset[train_size + val_size:]

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

## 4. Initialize the E(n) Equivariant Model

In [ ]:
from models.gnn_predictor import GNNPredictor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on {device}")

model = GNNPredictor(
    atom_feat_dim=65,    # RDKit atom features
    hidden_dim=128,      # EGNN hidden dimension
    n_layers=4,          # Deep equivariant messaging
    dropout=0.2          # For MC Dropout later
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
criterion = torch.nn.MSELoss()

## 5. Training Loop
We train for 50 epochs, saving the best checkpoint based on validation loss.

In [ ]:
import copy
from tqdm.auto import tqdm

best_val_loss = float('inf')
best_model_weights = None

EPOCHS = 50

for epoch in range(1, EPOCHS + 1):
    # --- TRAIN --- 
    model.train()
    train_loss = 0
    for data in tqdm(train_loader, desc=f"Epoch {epoch} Train"):
        data = data.to(device)
        optimizer.zero_grad()
        
        # We mock the protein cross-attention in the forward pass here if ESM is missing,
        # but the architecture expects data.x, data.edge_index, data.batch, data.pos
        out = model(data.x.float(), data.pos.float(), data.edge_index, data.batch, protein_embs=None)
        
        loss = criterion(out.squeeze(), data.y.float())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * data.num_graphs
    
    train_loss /= len(train_loader.dataset)
    
    # --- VALIDATION ---
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for data in val_loader:
            data = data.to(device)
            out = model(data.x.float(), data.pos.float(), data.edge_index, data.batch, protein_embs=None)
            loss = criterion(out.squeeze(), data.y.float())
            val_loss += loss.item() * data.num_graphs
            
    val_loss /= len(val_loader.dataset)
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch:03d} | Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_weights = copy.deepcopy(model.state_dict())

print("Training Complete!")

## 6. Export Trained Weights

In [ ]:
torch.save(best_model_weights, 'gnn_predictor_trained.pth')
print("Saved best weights to gnn_predictor_trained.pth")

from google.colab import files
files.download('gnn_predictor_trained.pth')